# GN Model Validation — Paper Reproducibility & Carena 2012 Fig. 5

Validate the user's GN engine against the two supplied GN-model papers, reproduce Carena et al. (JLT 2012) Fig. 5, and compare a representative subset with `EGN_model/EGN_adaptive.py`.

Paper curve/marker values are digitized from the supplied PDF and therefore carry digitization uncertainty.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import SVG, display
cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'2026_KICS_Fall_10').exists()),None)
if ROOT is None and Path('/content').exists():
    os.chdir('/content')
    if not Path('LSCNS').exists(): subprocess.run(['git','clone','https://github.com/kimheeseo/LSCNS.git'],check=True)
    ROOT=Path('/content/LSCNS')
if ROOT is None: raise RuntimeError('Could not locate LSCNS repository root')
GN_DIR=ROOT/'2026_KICS_Fall_10'/'GN_model'
RES=GN_DIR/'results'
print('Repository:',ROOT)
print('Results:',RES)


## 1. Paper 1 — equation-level reproducibility

In [ ]:
p1=json.loads((RES/'paper1_reproducibility.json').read_text())
display(pd.DataFrame(p1['equation_level_tests']))
display(pd.DataFrame(p1['qmc_convergence']))
print('Assessment:',p1['assessment'])
print('Max equation error [%]:',p1['max_equation_error_pct'])


## 2. Carena 2012 Figure 5 — paper vs GN code

In [ ]:
g=pd.read_csv(RES/'figure5_reproduction.csv')
display(g[['fiber','modulation','spacing_GHz','net_SE_bit_s_Hz','paper_Lmax_km','gn_Lmax_km','gn_error_pct']])
print('GN MAPE [%]:',g.gn_error_pct.mean())
print('GN median APE [%]:',g.gn_error_pct.median())
print('GN max APE [%]:',g.gn_error_pct.max())


## 3. Paper vs GN vs EGN

Carena-2012 Fig. 5 is a **GN-model** benchmark and its model curves use incoherent span accumulation. `EGN_adaptive.py`'s native full path instead uses rectangular spectra and coherent multi-span EGN physics. Therefore the three-way graph uses a transparent **paper-convention proxy** for representative 50-GHz QPSK points on PSCF, SMF and NZDSF: full one-span EGN NLI is calculated by `EGN_adaptive.py`, then accumulated incoherently across spans like Fig. 5. The EGN error is shown for comparison, but it is not treated as a native full-EGN validation score.

In [ ]:
e=pd.read_csv(RES/'paper_gn_egn_comparison.csv')
display(e[['fiber','modulation','spacing_GHz','paper_Lmax_km','gn_Lmax_km','egn_Lmax_km','gn_error_pct','egn_error_pct','egn_to_gn_nli_ratio_db']])
print('Same-subset GN MAPE [%]:',e.gn_error_pct.mean())
print('EGN MAPE [%]:',e.egn_error_pct.mean())
display(SVG(filename=str(RES/'paper_gn_egn_parity.svg')))


## 4. Machine-readable summary and limitations

In [ ]:
summary=json.loads((RES/'summary.json').read_text())
print(json.dumps(summary,indent=2))


## 5. Optional: recompute from scratch in Colab

Set `RUN_HEAVY=True` to rerun the numerical GN and full-EGN calculations.

In [ ]:
RUN_HEAVY=False
if RUN_HEAVY:
    subprocess.run([sys.executable,str(GN_DIR/'paper1_reproducibility.py')],cwd=ROOT,check=True)
    subprocess.run([sys.executable,str(GN_DIR/'figure5_benchmark.py')],cwd=ROOT,check=True)
    print('Recalculation complete.')
else:
    print('Using committed execution results. Set RUN_HEAVY=True to recompute.')
